In [65]:
import numpy as np
import pandas as pd
import config
import matplotlib.pyplot as plt
from scipy.stats import chisquare
# from scipy.stats import shapiro
# from scipy.stats import kruskal

from get_data import DataLoader

# Load data
DataLoader = DataLoader(config.DATA_PATH)
df = DataLoader.load_data()

df_2018pop = df[df['year']==2018].copy()
df_2018pop = df_2018pop[['who_region', 'pm25_concentration', 'population']].dropna()

Categories from Assignment 1

In [56]:
df_2018pop.loc[(df_2018pop['pm25_concentration'] >= 30), 'pm25_cats'] = '(6) 30 +'
df_2018pop.loc[(df_2018pop['pm25_concentration'] >= 25) & (df_2018pop['pm25_concentration'] < 30), 'pm25_cats'] = '(5) 25 - 30'
df_2018pop.loc[(df_2018pop['pm25_concentration'] >= 15) & (df_2018pop['pm25_concentration'] < 25), 'pm25_cats'] = '(4) 15 - 25'
df_2018pop.loc[(df_2018pop['pm25_concentration'] >= 10) & (df_2018pop['pm25_concentration'] < 15), 'pm25_cats'] = '(3) 10 - 15'
df_2018pop.loc[(df_2018pop['pm25_concentration'] >=  5) & (df_2018pop['pm25_concentration'] < 10), 'pm25_cats'] =  '(2) 5 - 10'
df_2018pop.loc[(df_2018pop['pm25_concentration'] <   5), 'pm25_cats'] = '(1) 5 -'

df_2018pop['highest_exposure'] = 0
df_2018pop.loc[(df_2018pop['pm25_concentration'] >= 30), 'highest_exposure'] = 1


Population bins from Assignment 1

In [57]:
df_pop = df_2018pop.dropna(subset='population').reset_index(drop=True)
pop = df_pop['population'].astype(int)

bins = list(range(0, pop.max(), 500000))
labels = [f'{bins[i]}-{bins[i+1]}' for i in range(len(bins)-1)]
df_2018pop['pop_binned'] = pd.cut(df_2018pop['population'], bins=bins, labels=labels, right=False)

In [58]:
# Step 4: Only use the first 3 population size categories

# Step 5: Group and create tables
df_2018pop2 = df_2018pop.groupby(['pop_binned', 'who_region', 'highest_exposure'],observed=False).size().reset_index(name='count')
df_2018pop2 = df_2018pop2[df_2018pop2['pop_binned'].isin(['0-500000', '500000-1000000', '1000000-1500000'])].reset_index(drop=True)

In [59]:
df_500000  = df_2018pop2[df_2018pop2['pop_binned']== '0-500000']
df_1000000 = df_2018pop2[df_2018pop2['pop_binned']== '500000-1000000']
df_1500000 = df_2018pop2[df_2018pop2['pop_binned']== '1000000-1500000']

In [60]:
pivot_table = df_2018pop2.pivot_table(
    index=['pop_binned', 'who_region'],
    columns='highest_exposure',
    values='count',
    observed=False,
    fill_value=0
).reset_index()

pivot_table['prevalence'] = pivot_table[1] / (pivot_table[0] + pivot_table[1])
pivot_table['prevalence'] = pivot_table['prevalence'].fillna(0)
pivot_table['prevalence'] = pivot_table['prevalence'].replace([np.inf, -np.inf], 0)
pivot_table['prevalence'] = pivot_table['prevalence'].replace([np.nan], 0)
pivot_table['prevalence'] = pivot_table['prevalence'].astype(float).round(3)

pivot_table[0] = pivot_table[0].astype(int)
pivot_table[1] = pivot_table[1].astype(int)

In [61]:
df_500000  = pivot_table[pivot_table['pop_binned']== '0-500000']
df_1000000 = pivot_table[pivot_table['pop_binned']== '500000-1000000']
df_1500000 = pivot_table[pivot_table['pop_binned']=='1000000-1500000']

In [62]:
print(df_500000.to_latex(index=False))

\begin{tabular}{llrrr}
\toprule
pop_binned & who_region & 0 & 1 & prevalence \\
\midrule
0-500000 & 1_Afr & 0 & 0 & 0.000000 \\
0-500000 & 2_Amr & 138 & 0 & 0.000000 \\
0-500000 & 3_Sear & 0 & 0 & 0.000000 \\
0-500000 & 4_Eur & 195 & 7 & 0.035000 \\
0-500000 & 5_Emr & 0 & 0 & 0.000000 \\
0-500000 & 7_NonMS & 0 & 0 & 0.000000 \\
\bottomrule
\end{tabular}



In [63]:
print(df_1000000.to_latex(index=False))

\begin{tabular}{llrrr}
\toprule
pop_binned & who_region & 0 & 1 & prevalence \\
\midrule
500000-1000000 & 1_Afr & 0 & 0 & 0.000000 \\
500000-1000000 & 2_Amr & 6 & 0 & 0.000000 \\
500000-1000000 & 3_Sear & 0 & 0 & 0.000000 \\
500000-1000000 & 4_Eur & 24 & 1 & 0.040000 \\
500000-1000000 & 5_Emr & 0 & 0 & 0.000000 \\
500000-1000000 & 7_NonMS & 0 & 0 & 0.000000 \\
\bottomrule
\end{tabular}



In [64]:
print(df_1500000.to_latex(index=False))

\begin{tabular}{llrrr}
\toprule
pop_binned & who_region & 0 & 1 & prevalence \\
\midrule
1000000-1500000 & 1_Afr & 0 & 0 & 0.000000 \\
1000000-1500000 & 2_Amr & 4 & 0 & 0.000000 \\
1000000-1500000 & 3_Sear & 0 & 0 & 0.000000 \\
1000000-1500000 & 4_Eur & 5 & 1 & 0.167000 \\
1000000-1500000 & 5_Emr & 0 & 1 & 1.000000 \\
1000000-1500000 & 7_NonMS & 0 & 0 & 0.000000 \\
\bottomrule
\end{tabular}



Just using prevalence for Europe since Africa has no data in the categories under investigation

* $H_0$ : The distribution between high exposure and no exposure is as expected (assuming equal distribution)
* $H_A$ : The distribution between high exposure and no exposure is not as expected

In [ ]:
# [high exposure, no exposure]
observed_0_500k  = [7, 195] 
observed_500k_1M = [1, 24] 
observed_1M_1_5M = [1, 5] 

expected_0_500k = [sum(observed_0_500k)/2, sum(observed_0_500k)/2]
expected_500k_1M = [sum(observed_500k_1M)/2, sum(observed_500k_1M)/2]
expected_1M_1_5M = [sum(observed_1M_1_5M)/2, sum(observed_1M_1_5M)/2]


chi2_stat_0_500k, p_value_0_500k   = chisquare(f_obs=observed_0_500k, f_exp=expected_0_500k)
chi2_stat_500k_1M, p_value_500k_1M = chisquare(f_obs=observed_500k_1M, f_exp=expected_500k_1M)
chi2_stat_1M_1_5M, p_value_1M_1_5M = chisquare(f_obs=observed_1M_1_5M, f_exp=expected_1M_1_5M)

print(f"Chi-squared test results:")
print(f"0-500k: Chi2 Stat = {chi2_stat_0_500k}, p-value = {p_value_0_500k}")
print(f"500k-1M: Chi2 Stat = {chi2_stat_500k_1M}, p-value = {p_value_500k_1M}")
print(f"1M-1.5M: Chi2 Stat = {chi2_stat_1M_1_5M}, p-value = {p_value_1M_1_5M}")

Chi-squared test results:
0-500k: Chi2 Stat = 174.97029702970298, p-value = 6.0770533357917055e-40
500k-1M: Chi2 Stat = 21.16, p-value = 4.224909405005696e-06
1M-1.5M: Chi2 Stat = 2.6666666666666665, p-value = 0.10247043485974942
